# Notebook 01 of 7 — Getting Started + Providers (Track B: Free-only)

*Portfolio Intelligence Engine — User Guide Series, **Track B (free sources only)**.*
[Series README](../portfolio/README.md) · [Story Bible](../portfolio/STORY_BIBLE.md) · Epic
[#1428](https://github.com/prajoria/OpenBB/issues/1428) · This notebook
[#1437](https://github.com/prajoria/OpenBB/issues/1437) · Track A counterpart:
[`../portfolio/01-getting-started-and-providers.ipynb`](../portfolio/01-getting-started-and-providers.ipynb).

---

## Where we are in Sam's story

This is the **free-only** mirror. Sam is 14 months in, up 4% while SPY is up 22%, and wants to know if the platform is honest without paying anyone. No FMP key required in this notebook — the whole series' Track B lane runs on `cboe` (the listing exchange), `sec` (EDGAR), and — as a last resort for endpoints neither authoritative tier covers — recorded `yfinance` snapshots.

By the end of this notebook we will be able to answer one question:

> *What does this system give me access to without a paid data key, and where does the free path end?*


## 0. Before we run anything

The whole series lives in an isolated Python environment called
`.venv_portfolio`. That's deliberate — the fork has a parallel lane
(techtrade) that shares its own venv, and mixing them causes silent
version conflicts. If the cell below halts, follow the printed setup
command exactly, then come back.

*The code cell below asserts we're in the right interpreter and creates
the shared state directory the later notebooks will write into.*


In [ ]:
# [Track B / NB01 §0] environment sanity — assert .venv_portfolio + create STATE dir
import sys, pathlib

assert "venv_portfolio" in sys.executable, (
    "Portfolio notebooks require .venv_portfolio, not the current interpreter.\n"
    "From repo root:\n"
    "  .\\.venv_portfolio\\Scripts\\Activate.ps1\n"
    "  python -m ipykernel install --user --name openbb-portfolio\n"
    f"Currently running: {sys.executable}"
)
STATE = pathlib.Path(".notebook_state")
STATE.mkdir(exist_ok=True)

print(f"Python:                   {sys.version.split()[0]}")
print(f"Interpreter (basename):   {pathlib.Path(sys.executable).name}")
print(f"venv sanity check:        passed (interpreter path contains 'venv_portfolio')")
print(f"State dir (repo-rel):     {STATE}/")


Python:                   3.12.10
Interpreter (basename):   python.exe
venv sanity check:        passed (interpreter path contains 'venv_portfolio')
State dir (repo-rel):     .notebook_state/


## 0.5 Why free-only? Provider vocabulary for this notebook

This notebook is the **Track B mirror** of the Track A NB01. Track A
(`../portfolio/01-getting-started-and-providers.ipynb`) uses `fmp_cached`
as its primary tier because a paid FMP key is the fastest way to get
broker-quality prices, fundamentals, ratings, and calendars in one
place. Track B is what the series looks like when you have **no paid
keys at all** — only public, first-party sources.

The three-tier free chain we use throughout Track B:

1. **`cboe` — free-authoritative equity + options.** CBOE is the listing
   exchange most US options are quoted on. It exposes free quotes,
   EOD historical bars, and full options chains + IV surface. When
   Track B says "free-authoritative equity," this is who.
2. **`sec` — free-authoritative filings (EDGAR).** ETF/bond fund
   holdings (Form N-PORT), institutional-manager positions (Form 13F),
   insider transactions (Form 3/4/5), and XBRL company facts. All US
   government filings, so free and redistribution-safe.
3. **`yfinance` — LAST-RESORT, personal-use only.** Never a silent
   default. Only used through the offline snapshot store at
   `~/.scrape_record/snapshots.db` (per-operator, never committed) and
   only for endpoints neither `cboe` nor `sec` cover on my plan.

Provenance: epic [#1428](https://github.com/prajoria/OpenBB/issues/1428)
(authoritative-data-source), [#1425](https://github.com/prajoria/OpenBB/issues/1425)
(user-local Yahoo snapshot store), [#1426](https://github.com/prajoria/OpenBB/issues/1426)
(N-PORT look-through consumer).


## 1. The `obb` object

Everything in the platform hangs off one Python object called `obb`. Two
things live on it:

- **Extensions** — domain areas (`equity`, `etf`, `derivatives`,
  `portfolio_intel`, and so on). Each exposes commands you call like a
  normal Python function.
- **Providers** — data sources. In Track B we restrict ourselves to
  `cboe`, `sec`, and `yfinance` (last-resort). The same extension
  command can be routed to any provider that implements it.

The mental model: an extension is a *question* ("give me this ETF's
holdings"), a provider is *whoever answers* ("here's what EDGAR
filed"). Same question, potentially many answerers — chosen by a
priority order you control.

*The code cell below prints the extensions loaded in this venv and the
fetcher count for each of the three Track B providers.*


In [ ]:
# [Track B / NB01 §1] extensions on obb + free-only fetcher counts
from openbb import obb
from openbb_cboe import cboe_provider
from openbb_sec import sec_provider
from openbb_yfinance import yfinance_provider

extensions = sorted(a for a in dir(obb) if not a.startswith("_"))
print(f"Extensions loaded ({len(extensions)}):")
for e in extensions:
    print(f"  obb.{e}")

print()
print("Registered fetchers per Track B provider (free-authoritative -> last-resort):")
print(f"  cboe:       {len(cboe_provider.fetcher_dict):>4}   (listing exchange -- free-authoritative quotes/options/history)")
print(f"  sec:        {len(sec_provider.fetcher_dict):>4}   (SEC EDGAR -- free-authoritative filings: N-PORT, 13F, Form 4, XBRL)")
print(f"  yfinance:   {len(yfinance_provider.fetcher_dict):>4}   (Yahoo -- LAST-RESORT, personal-use only, never a silent default)")


Extensions loaded (22):
  obb.backtest
  obb.cftc
  obb.commodity
  obb.coverage
  obb.crypto
  obb.currency
  obb.derivatives
  obb.economy
  obb.equity
  obb.etf
  obb.fixedincome
  obb.imf_utils
  obb.index
  obb.news
  obb.portfolio_intel
  obb.reference
  obb.regime
  obb.regulators
  obb.system
  obb.techtrade
  obb.uscongress
  obb.user

Registered fetchers per Track B provider (free-authoritative -> last-resort):
  cboe:         11   (listing exchange -- free-authoritative quotes/options/history)
  sec:          29   (SEC EDGAR -- free-authoritative filings: N-PORT, 13F, Form 4, XBRL)
  yfinance:     35   (Yahoo -- LAST-RESORT, personal-use only, never a silent default)


## 2. The three-tier free provider chain

Track B resolves every provider call in a **three-tier priority chain**:

1. **Free-authoritative equity/options — `cboe`.** Quotes
   (`EquityQuote`), index/ETF/equity historical bars, options chains
   + IV surface. Free, redistribution-safe, first-party. **Caveat:**
   the free CBOE feed is end-of-day; there's no intraday tick in this
   lane.
2. **Free-authoritative filings — `sec` (SEC EDGAR).** N-PORT
   (ETF/bond fund holdings, quarterly), 13F (institutional-manager
   positions), Form 3/4/5 (insider transactions), XBRL company facts
   (fundamentals). Authoritative because the issuer filed it.
   **Caveat:** N-PORT is quarter-visible with a ~30–60 day lag —
   correct for concentration and look-through, wrong for intraday.
3. **Last resort — `yfinance` (personal-use only).** Used only for
   endpoints `cboe` and `sec` do not cover on my plan, and only through
   the snapshot-backed offline reader at `~/.scrape_record/snapshots.db`.
   Never a silent default; every call site is labelled *last-resort*.

### Capability → free-authoritative source

| Capability | Track B source |
|---|---|
| Quotes | `cboe` |
| Prices / EOD bars / index history | `cboe` |
| Options chains + IV context | `cboe` |
| ETF / bond-fund holdings (look-through) | `sec` (N-PORT) |
| Institutional holdings (smart money) | `sec` (Form 13F) |
| Insider transactions | `sec` (Form 3/4/5) |
| Fundamentals / statements | `sec` (XBRL income/balance/cash) |
| Analyst ratings / price targets | **no free authoritative source — documented gap** |
| Anything the three tiers miss | snapshot-backed `yfinance` (personal-use only) |

The cell below hits the top of each tier on MSFT / QQQ so you can see
the actual response shape.


In [ ]:
# [Track B / NB01 §2] Walk down the free tier chain on a real call.
from openbb import obb
import warnings; warnings.filterwarnings("ignore")

# Tier 1 -- cboe (listing exchange, free-authoritative EOD quote + IV context)
print("Tier 1 -- cboe (listing exchange, free-authoritative):")
q = obb.equity.price.quote(symbol="MSFT", provider="cboe")
row = q.to_df().iloc[0]
print(f"  provider=cboe  MSFT last_price={row['last_price']}  bid={row['bid']}  ask={row['ask']}")
print(f"  IV context: iv30={row['iv30']:.4f}  hv30_annual_high={row['hv30_annual_high']:.4f}")
print(f"  last_timestamp={row['last_timestamp']}   (CBOE free tier is end-of-day)")
print()

# Tier 2 -- sec (EDGAR, N-PORT for ETF look-through)
print("Tier 2 -- obb.etf.nport_disclosure (SEC Form N-PORT):")
nport = obb.etf.nport_disclosure(symbol="QQQ", provider="sec")
rows = nport.results
top5 = sorted(rows, key=lambda r: -(r.weight or 0))[:5]
print(f"  provider=sec  rows={len(rows)}  (QQQ latest N-PORT filing)")
print("  Top 5 holdings by weight:")
for r in top5:
    print(f"    {(r.name or '')[:34]:<36} {(r.weight or 0)*100:>6.2f}%   CUSIP {r.cusip}")
print(f"  (Caveat: N-PORT is quarter-visible with ~30-60 day lag.)")
print()

# Tier 3 -- yfinance snapshot (LAST-RESORT, personal-use only).
print("Tier 3 -- YFinanceEquityQuoteRecorded (LAST-RESORT, offline snapshot):")
try:
    from openbb_yfinance.models.recorded_equity_quote import (
        YFinanceEquityQuoteRecordedFetcher,
    )
    r = YFinanceEquityQuoteRecordedFetcher.fetch_from_snapshot("MSFT")
    print(f"  provider=yfinance(snapshot)  symbol={r.symbol}  last_price={r.last_price}  captured_at={r.captured_at}")
except Exception as exc:
    print(f"  (no local MSFT snapshot recorded -- {type(exc).__name__})")
    print(f"  Record with: scrape-record record yahoo_equity_quote --symbol MSFT")
print("  (snapshot layer read from ~/.scrape_record/snapshots.db -- never live)")


Tier 1 -- cboe (listing exchange, free-authoritative):


  provider=cboe  MSFT last_price=381.4  bid=381.35  ask=381.4
  IV context: iv30=0.4214  hv30_annual_high=0.4355
  last_timestamp=2026-07-24 15:59:59   (CBOE free tier is end-of-day)

Tier 2 -- obb.etf.nport_disclosure (SEC Form N-PORT):
  provider=sec  rows=102  (QQQ latest N-PORT filing)
  Top 5 holdings by weight:
    NVIDIA Corp.                           8.68%   CUSIP 67066G104
    Apple Inc.                             7.63%   CUSIP 037833100
    Microsoft Corp.                        5.63%   CUSIP 594918104
    Amazon.com, Inc.                       4.58%   CUSIP 023135106
    Tesla, Inc.                            3.80%   CUSIP 88160R101
  (Caveat: N-PORT is quarter-visible with ~30-60 day lag.)

Tier 3 -- YFinanceEquityQuoteRecorded (LAST-RESORT, offline snapshot):
  (no local MSFT snapshot recorded -- EmptyDataError)
  Record with: scrape-record record yahoo_equity_quote --symbol MSFT
  (snapshot layer read from ~/.scrape_record/snapshots.db -- never live)


## 3. The free-authoritative fetchers that will actually matter

Not everything Track A leans on has a free-authoritative equivalent.
The table below is the free-only subset that will come back in later
Track B notebooks, teased now so nothing surprises you later.

**Prices + IV context (`cboe`):**
`EquityQuote`, `EquityHistorical`, `CboeOptionsChains` (via
`obb.derivatives.options.chains`).

**Filings (`sec`):**
`NportDisclosure` (ETF/bond look-through), `Form13FHoldings` (smart
money), `InsiderTrading` (Form 4), XBRL fundamentals
(`IncomeStatement`, `BalanceSheet`, `CashFlowStatement`).

**Documented gaps in Track B** (Track A gets these from `fmp_cached`):

- **Analyst ratings / price targets** — no free authoritative source.
  We omit them in Track B and call out the gap in prose wherever a
  Track A analysis leans on them.
- **Realtime / intraday quotes** — CBOE free tier is EOD only.
- **`EtfHoldings` classic shape (weights table)** — replaced by
  N-PORT in this series (GH [#1426](https://github.com/prajoria/OpenBB/issues/1426)).

*The code cell below shape-checks each free fetcher with a one-line
call.*


In [ ]:
# [Track B / NB01 §3] shape check on the free fetchers that come back in later notebooks
from openbb import obb
import warnings; warnings.filterwarnings("ignore")

CHECKS = [
    ("EquityQuote (cboe)",           lambda: obb.equity.price.quote(symbol="MSFT", provider="cboe")),
    ("EquityHistorical 5d (cboe)",   lambda: obb.equity.price.historical(symbol="MSFT", provider="cboe", start_date="2026-07-14", end_date="2026-07-24")),
    ("OptionsChains (cboe)",         lambda: obb.derivatives.options.chains(symbol="MSFT", provider="cboe")),
    ("NportDisclosure QQQ (sec)",    lambda: obb.etf.nport_disclosure(symbol="QQQ", provider="sec")),
    ("InsiderTrading MSFT (sec)",    lambda: obb.equity.ownership.insider_trading(symbol="MSFT", provider="sec", limit=5)),
    ("IncomeStatement MSFT (sec)",   lambda: obb.equity.fundamental.income(symbol="MSFT", provider="sec", limit=1)),
    ("BalanceSheet MSFT (sec)",      lambda: obb.equity.fundamental.balance(symbol="MSFT", provider="sec", limit=1)),
    ("CashFlow MSFT (sec)",          lambda: obb.equity.fundamental.cash(symbol="MSFT", provider="sec", limit=1)),
]

print(f"{'Fetcher':<34}{'Rows':>6}   Provider    Status")
print("-" * 72)
for label, fn in CHECKS:
    try:
        result = fn()
        # Some result models don't have to_df; fall back to .results length.
        try:
            n = len(result.to_df())
        except Exception:
            n = len(result.results) if hasattr(result, "results") else "?"
        prov = "cboe" if "cboe" in label else "sec"
        print(f"{label:<34}{n:>6}   {prov:<10}  ok")
    except Exception as exc:
        prov = "cboe" if "cboe" in label else "sec"
        print(f"{label:<34}{'-':>6}   {prov:<10}  {type(exc).__name__}: {str(exc)[:24]}")

# Documented gap -- analyst ratings / price targets have no free authoritative source.
print()
print("Documented gap: analyst ratings / price targets")
print("  Track A calls obb.equity.estimates.consensus with the paid FMP tier;")
print("  Track B has no free-authoritative equivalent. Section is omitted in Track B NB02/NB08.")


Fetcher                             Rows   Provider    Status
------------------------------------------------------------------------


EquityQuote (cboe)                     1   cboe        ok


EquityHistorical 5d (cboe)             8   cboe        ok


OptionsChains (cboe)                3462   cboe        ok
NportDisclosure QQQ (sec)            102   sec         ok



Found 5 total filings and 0 uncached entries to download, estimated download time: 0 seconds.



InsiderTrading MSFT (sec)              6   sec         ok


IncomeStatement MSFT (sec)             1   sec         ok


BalanceSheet MSFT (sec)                1   sec         ok


CashFlow MSFT (sec)                    1   sec         ok

Documented gap: analyst ratings / price targets
  Track A calls obb.equity.estimates.consensus with the paid FMP tier;
  Track B has no free-authoritative equivalent. Section is omitted in Track B NB02/NB08.


## 4. The offline-snapshot pattern (yfinance without hitting yfinance)

If you read Track A NB01 §4 already, this is the same layer — I'll keep
it short. The fork ships a small framework called `scrape_record` that
records a page once and replays it from a user-local SQLite DB at
`~/.scrape_record/snapshots.db`. Nothing Yahoo-shaped ships in git;
each operator keeps their own local copy (see GH [#1425](https://github.com/prajoria/OpenBB/issues/1425) for the ToS
rationale — this mirrors how `yfinance` itself is documented, "personal
use only").

In Track B the snapshot layer covers the specific things neither `cboe`
nor `sec` covers on the free plan:

- `YFinanceEquityQuoteRecordedFetcher` — freshness-convenience fallback
  for equity quote when we've cached one
- `YFinanceEquityInfoRecordedFetcher` — sector/industry/summary
- `YFinanceRecordedOptionsChainsFetcher` — full options chain per
  expiry (redundant with CBOE for major names; kept for endpoints CBOE
  doesn't expose)
- `YFinanceAtmIvTermStructureFetcher` — ATM IV curve

If a symbol isn't present locally the fetcher raises
`FileNotFoundError` and prints the exact record command. **No
live-scraping surprises at query time. No automation ever hits Yahoo
live.**

*The cell below demonstrates the pattern on `EquityQuote`.*


In [ ]:
# [Track B / NB01 §4] Snapshot-backed yfinance demo -- last-resort tier only.
try:
    from openbb_yfinance.models.recorded_equity_quote import (
        YFinanceEquityQuoteRecordedFetcher,
    )
    row = YFinanceEquityQuoteRecordedFetcher.fetch_from_snapshot("MSFT")
    print(f"Symbol:                {row.symbol}")
    print(f"Last price (snapshot): {row.last_price}")
    print(f"Market cap:            {row.market_cap:,}" if row.market_cap else "Market cap:            (not in snapshot)")
    print(f"Captured at:           {row.captured_at}")
except Exception as exc:
    print(f"(no local MSFT snapshot recorded -- {type(exc).__name__})")
    print(f"Record with: scrape-record record yahoo_equity_quote --symbol MSFT")
print()
print("Source (per-operator, never committed): ~/.scrape_record/snapshots.db")
print("Record / refresh: scrape-record record yahoo_equity_quote --symbol MSFT")
print()
print("For ETF/bond look-through Track B uses SEC N-PORT (see Section 2), not yahoo_etf_holdings.")


(no local MSFT snapshot recorded -- EmptyDataError)
Record with: scrape-record record yahoo_equity_quote --symbol MSFT

Source (per-operator, never committed): ~/.scrape_record/snapshots.db
Record / refresh: scrape-record record yahoo_equity_quote --symbol MSFT

For ETF/bond look-through Track B uses SEC N-PORT (see Section 2), not yahoo_etf_holdings.


## 5. The through-line basket

From NB03 onward we work against the same fixed basket used in Track A:

| Ticker | Weight |
|--------|--------|
| MSFT   | 12% |
| NVDA   | 10% |
| GOOGL  | 8% |
| AAPL   | 8% |
| AMD    | 6% |
| QQQ    | 15% |
| VTI    | 20% |
| VNQ    | 8% |
| BND    | 10% |
| GLD    | 3% |

Weights are round-number synthetic. This is not anyone's real portfolio.

**Note:** the basket file is intentionally **shared with Track A** —
both tracks consume `.notebook_state/basket.json`. That way the
free-only lane produces the same concentration story Track A does; only
the data path differs. If you run Track A NB01 §5 after this cell, it
will simply overwrite the identical bytes.

Look at that list. Ten different things, five of them mega-cap tech,
and three ETFs that "diversify" it. It *looks* diversified. Wait for
NB03.

*The code cell below writes the basket to `.notebook_state/basket.json`.*


In [ ]:
# [Track B / NB01 §5] write the through-line basket to .notebook_state/basket.json
import json
from pathlib import Path

BASKET = [
    {"ticker": "MSFT",  "weight": 0.12, "kind": "equity", "note": "single-name deep-dive subject (NB02)"},
    {"ticker": "NVDA",  "weight": 0.10, "kind": "equity", "note": "semi cluster"},
    {"ticker": "GOOGL", "weight": 0.08, "kind": "equity", "note": "mega-cap tech"},
    {"ticker": "AAPL",  "weight": 0.08, "kind": "equity", "note": "mega-cap tech"},
    {"ticker": "AMD",   "weight": 0.06, "kind": "equity", "note": "semi peer"},
    {"ticker": "QQQ",   "weight": 0.15, "kind": "etf",    "note": "top holdings = the tickers above"},
    {"ticker": "VTI",   "weight": 0.20, "kind": "etf",    "note": "broad market, ~30% tech under the hood"},
    {"ticker": "VNQ",   "weight": 0.08, "kind": "etf",    "note": "REIT -- distinct sector"},
    {"ticker": "BND",   "weight": 0.10, "kind": "etf",    "note": "bond ETF, BondLadder demo target"},
    {"ticker": "GLD",   "weight": 0.03, "kind": "etf",    "note": "tail hedge"},
]

out = Path(".notebook_state/basket.json")
out.parent.mkdir(exist_ok=True)
out.write_text(json.dumps(BASKET, indent=2), encoding="utf-8")

total = sum(p["weight"] for p in BASKET)
print(f"Wrote (repo-rel): {out}")
print(f"Positions: {len(BASKET)}    Total weight: {total*100:.1f}%")
print(f"(Shared with Track A -- both tracks consume this file.)")
print()
print(f"{'Ticker':<8}{'Weight':>8}   Kind    Note")
print("-" * 78)
for p in BASKET:
    print(f"{p['ticker']:<8}{p['weight']*100:>7.1f}%   {p['kind']:<6}  {p['note']}")


Wrote (repo-rel): .notebook_state\basket.json
Positions: 10    Total weight: 100.0%
(Shared with Track A -- both tracks consume this file.)

Ticker    Weight   Kind    Note
------------------------------------------------------------------------------
MSFT       12.0%   equity  single-name deep-dive subject (NB02)
NVDA       10.0%   equity  semi cluster
GOOGL       8.0%   equity  mega-cap tech
AAPL        8.0%   equity  mega-cap tech
AMD         6.0%   equity  semi peer
QQQ        15.0%   etf     top holdings = the tickers above
VTI        20.0%   etf     broad market, ~30% tech under the hood
VNQ         8.0%   etf     REIT -- distinct sector
BND        10.0%   etf     bond ETF, BondLadder demo target
GLD         3.0%   etf     tail hedge


---

## What is NOT in this notebook

- **Real-time price feeds / order books.** CBOE free tier is EOD;
  Track A gets realtime via `fmp_cached`, Track B does not.
- **Analyst ratings / price targets.** No free authoritative source.
  Track A uses `fmp_cached` for these; Track B readers get a
  documented gap in NB02/NB08 wherever those numbers would land.
- **Alternative data (satellite, credit-card, dark-pool).** Free-tier
  only — none of these are in scope for either track.
- **MCP client demos.** The fork ships an MCP server; NB07 has a
  pointer.

## Preview of NB02

Plumbing works, free-tier constraints are on the table. Next up in
Track B NB02 we take MSFT — the largest single position — and run the
7-phase Analysis pipeline using the free chain wherever we can
(prices from CBOE, fundamentals from SEC XBRL). The composite still
lands a decision label; where the paid-only inputs (analyst grades,
price-target consensus) would go, Track B leaves an explicit gap
marker instead of a silent fallback.


## 📚 Further reading

The Investopedia links first cited in Track A NB01 are not re-linked
here (per the series' first-occurrence rule) — the bare terms below
are pointers. New references specific to the Track B free path are
linked in full.

**Already-linked terms** (see Track A NB01's Further Reading): market
data, quote, tick, OHLCV bar, adjusted close, stock split, dividend,
market capitalization, free float, ETF, CBOE, basket trade.

**Free-authoritative sources (new in Track B):**

- **SEC EDGAR (issuer filings)** — <https://www.sec.gov/edgar>
- **SEC Form N-PORT (fund holdings)** — <https://www.sec.gov/rules-regulations/investment-management-rulemaking/n-port-general>
- **SEC Form 13F (institutional holdings)** — <https://www.sec.gov/divisions/investment/13ffaq>
- **SEC Form 4 (insider transactions)** — <https://www.sec.gov/about/forms/form4data.pdf>
- **CBOE market data (options + IV)** — <https://www.cboe.com/us/options/market_statistics/>
- **`yfinance` project (personal-use rationale)** — <https://github.com/ranaroussi/yfinance>
